# Lesson 13: Text Representation — Bag-of-Words, N-Grams & TF-IDF

In this notebook, you will:
1. Vectorize text using **CountVectorizer** (BoW + Bigrams) and **TfidfVectorizer**.
2. Train a **PyTorch** linear classifier on TF-IDF features.

In [ ]:
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.datasets import fetch_20newsgroups

cats = ["sci.space", "rec.sport.baseball", "talk.politics.misc"]
train = fetch_20newsgroups(subset="train", categories=cats, remove=("headers", "footers", "quotes"))
test = fetch_20newsgroups(subset="test", categories=cats, remove=("headers", "footers", "quotes"))

vec = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), stop_words="english")
X_tr = torch.tensor(vec.fit_transform(train.data).toarray(), dtype=torch.float32)
y_tr = torch.tensor(train.target, dtype=torch.long)
X_te = torch.tensor(vec.transform(test.data).toarray(), dtype=torch.float32)
y_te = torch.tensor(test.target, dtype=torch.long)

model = nn.Linear(X_tr.shape[1], len(cats))
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(25):
    opt.zero_grad()
    loss = loss_fn(model(X_tr), y_tr)
    loss.backward()
    opt.step()

with torch.no_grad():
    acc = (model(X_te).argmax(1) == y_te).float().mean().item()
print(f"TF-IDF + PyTorch Linear Test Accuracy: {acc*100:.2f}%")

## Part 2: Inspecting Top Weighted N-Gram Features

In [ ]:
feature_names = vec.get_feature_names_out()
weights = model.weight.detach().numpy()
for idx, cat in enumerate(cats):
    top_idx = weights[idx].argsort()[-5:][::-1]
    print(f"{cat}: {[feature_names[i] for i in top_idx]}")